1.) Import Libraries + set up

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print("All libraries imported successfully!")

2. Data Preparation
    - loading dataset
    - preprocessing
    - creating sequences
    - splitting data

In [ ]:
# Import utility functions
from utils import generate_synthetic_bus_data

# Generate synthetic data
print("Generating synthetic bus delay dataset...")
df_raw = generate_synthetic_bus_data(n_samples=10000)

# Save to CSV
df_raw.to_csv("dummy_data/synthetic_bus_data.csv", index=False)
print(f"Dataset saved to dummy_data/synthetic_bus_data.csv")

# Display basic information
print(f"\nDataset shape: {df_raw.shape}")
print(f"\nColumn names and types:")
print(df_raw.dtypes)
print(f"\nFirst 5 rows:")
df_raw.head()

3. Feature Engineering
   - Create lag features (`create_lag_features`).  

These functions are defined in `utils.py` for reuse.

In [ ]:
# Import feature engineering functions
from utils import engineer_features

# Apply feature engineering
print("Applying feature engineering...")
df_engineered = engineer_features(df_raw)

print(f"\nOriginal features: {df_raw.shape[1]}")
print(f"After feature engineering: {df_engineered.shape[1]}")
print(f"New features added: {df_engineered.shape[1] - df_raw.shape[1]}")

print(f"\nAll features:")
print(df_engineered.columns.tolist())

# Display statistics
print(f"\nFeature statistics:")
df_engineered.describe()

4. Model Set Up
   - Train an XGBoost Regressor using lagged and engineered features.


In [ ]:
# Import data preparation and XGBoost
from utils import prepare_data_for_xgboost
from xgboost import XGBRegressor

# Prepare data for XGBoost
print("Preparing data for XGBoost...")
X_train, X_test, y_train, y_test, feature_names = prepare_data_for_xgboost(
    df_engineered, 
    target_col='tta_sec', 
    test_size=0.2, 
    random_state=42
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")
print(f"Number of features: {len(feature_names)}")

# Initialize XGBoost Regressor with optimized hyperparameters
print("\nInitializing XGBoost model...")
xgb_model = XGBRegressor(
    n_estimators=200,           # Number of boosting rounds
    max_depth=6,                # Maximum tree depth
    learning_rate=0.1,          # Step size shrinkage
    subsample=0.8,              # Fraction of samples for training each tree
    colsample_bytree=0.8,       # Fraction of features for training each tree
    min_child_weight=3,         # Minimum sum of instance weight in a child
    gamma=0.1,                  # Minimum loss reduction for split
    reg_alpha=0.01,             # L1 regularization
    reg_lambda=1.0,             # L2 regularization
    random_state=42,
    n_jobs=-1,                  # Use all CPU cores
    objective='reg:squarederror'
)

# Train the model
print("\nTraining XGBoost model...")
xgb_model.fit(
    X_train, 
    y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    eval_metric='rmse',
    verbose=50  # Print evaluation every 50 rounds
)

print("\nModel training completed!")

5. Evaluation/ Visualization
   - Compare true vs. predicted delay values for both models.
   - Compute evaluation metrics such as RMSE and R²
   - Plot predictions to visually assess performance


In [ ]:
# Import evaluation functions
from utils import evaluate_model, plot_predictions, plot_feature_importance

# Make predictions
print("Making predictions on test set...")
y_pred_train = xgb_model.predict(X_train)
y_pred_test = xgb_model.predict(X_test)

# Evaluate on training set
train_metrics = evaluate_model(y_train, y_pred_train, model_name="XGBoost (Training Set)")

# Evaluate on test set
test_metrics = evaluate_model(y_test, y_pred_test, model_name="XGBoost (Test Set)")

# Plot predictions
print("\nPlotting predictions...")
plot_predictions(y_test, y_pred_test, model_name="XGBoost", sample_size=500)

# Plot feature importance
print("\nPlotting feature importance...")
plot_feature_importance(xgb_model, feature_names, top_n=15)

# Print summary
print("\n" + "="*80)
print("MODEL SUMMARY")
print("="*80)
print(f"Training RMSE: {train_metrics['rmse']:.2f} seconds ({train_metrics['rmse']/60:.2f} minutes)")
print(f"Test RMSE:     {test_metrics['rmse']:.2f} seconds ({test_metrics['rmse']/60:.2f} minutes)")
print(f"Test MAE:      {test_metrics['mae']:.2f} seconds ({test_metrics['mae']/60:.2f} minutes)")
print(f"Test R²:       {test_metrics['r2']:.4f}")
print("="*80)